# %% [STEP 0 — Script Overview]
# ------------------------------------------------------------
# This script calculates window width (W) and height (H) for each
# dwelling based on total window area and built form.
#
# Key logic:
# - Total window area is divided across a fixed number of windows
# - Window dimensions are derived from area and aspect ratio
# - Aspect ratio depends on BUILT_FORM:
#     • Detached → 4:2 (W = 2H)
#     • All others → 3:2 (W = 1.5H)
#
# Output:
# - Adds WINDOW_AREA_SINGLE
# - Adds WINDOW_H (height)
# - Adds WINDOW_W (width)
# - Saves updated dataset to file
# ------------------------------------------------------------

In [21]:
# %% [STEP 1 — Import Libraries]
import pandas as pd
import math

In [22]:
# %% [STEP 2 — Load Dataset]
file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_window_area_update.csv"

df = pd.read_csv(file_path)

print("Dataset loaded")
print(df.shape)
df.head()

Dataset loaded
(117, 219)


,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,INSPECTION_DATE,TYPE_OF_ASSESSMENT,...,RIDGE_TO_ATTIC_HEIGHT,HALF_WALL_WIDTH,ROOF_WIDTH,HALF_ROOF_AREA,ROOF_AREA,USABLE_ROOF_LENGTH,USABLE_ROOF_WIDTH,PANELS_ALONG_LENGTH,PANELS_ALONG_WIDTH,POSSIBLE_PV_PANELS
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055692,-4.223337,9/29/25,"RdSAP, existing dwelling",...,2.291288,2.291288,3.240370,14.849242,29.698485,3.895189,2.754315,2,2,0
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.052824,-4.220640,9/19/25,"RdSAP, existing dwelling",...,3.240370,3.240370,4.582576,44.547727,89.095454,8.262944,3.895189,4,3,8
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055503,-4.223691,7/29/25,"RdSAP, existing dwelling",...,2.915476,2.915476,4.123106,24.041631,48.083261,4.956309,3.504640,2,3,2
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.058427,-4.224838,7/22/25,"RdSAP, existing dwelling",...,3.253204,3.253204,4.600725,44.901281,89.802561,8.295669,3.910616,4,3,8
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.054738,-4.228307,7/17/25,"RdSAP, existing dwelling",...,6.563332,4.163332,7.772429,97.077613,194.155227,10.616497,6.606565,5,5,21


In [23]:
# %% [STEP 3 — Define Constants]
TOTAL_WINDOWS = 7

In [24]:
# %% [STEP 4 — Window Area Per Window]
df["WINDOW_AREA_SINGLE"] = df["WINDOW_AREA"] / TOTAL_WINDOWS

In [ ]:
# %% [STEP 5 — Define Aspect Ratio Function]
def get_aspect_ratio(built_form):
    if built_form == "Detached":
        return 2.25  # W = 2H
    else:
        return 1.75  # W = 1.5H

In [26]:
# %% [STEP 6 — Assign Aspect Ratio Per Row]
df["ASPECT_RATIO"] = df["BUILT_FORM"].apply(get_aspect_ratio)

In [27]:
# %% [STEP 7 — Calculate Window Height]
# From: Area = W * H = (ratio * H) * H = ratio * H^2
# Therefore: H = sqrt(Area / ratio)

df["WINDOW_H"] = (df["WINDOW_AREA_SINGLE"] / df["ASPECT_RATIO"]) ** 0.5

In [28]:
# %% [STEP 8 — Calculate Window Width]
df["WINDOW_W"] = df["ASPECT_RATIO"] * df["WINDOW_H"]

In [29]:
# %% [STEP 9 — Inspect Results]
df[[
    "BUILT_FORM",
    "WINDOW_AREA",
    "WINDOW_AREA_SINGLE",
    "ASPECT_RATIO",
    "WINDOW_W",
    "WINDOW_H"
]].head()

,BUILT_FORM,WINDOW_AREA,WINDOW_AREA_SINGLE,ASPECT_RATIO,WINDOW_W,WINDOW_H
0,Semi-Detached,12.5358,1.790829,1.75,1.770297,1.011598
1,Detached,22.3276,3.189657,2.25,2.678942,1.190641
2,End-Terrace,14.3142,2.044886,1.75,1.891706,1.080975
3,Detached,23.0673,3.295329,2.25,2.722956,1.210203
4,Detached,19.3444,2.763486,2.25,2.493560,1.108249


In [30]:
# %% [STEP 10 — Save Updated Dataset]
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_window_dimensions_update.csv"

df.to_csv(output_path, index=False)

print("Dataset saved to:")
print(output_path)

Dataset saved to:
/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_window_dimensions_update.csv
